# Nested Learning for CAD — chặng 1
W1/W3/W4 synthetic correctness, exact 16-event resource gate, pinned upstream parity. Không cần MVTec hoặc ViT checkpoint. Không chạy benchmark/anomaly test. Hai T4 là hai GPU riêng; notebook dùng một GPU.

Upload `stage1_source_bundle.zip` thành Kaggle Dataset và attach vào notebook. Kaggle có thể giữ file ZIP hoặc tự bung thành cây thư mục; bootstrap hỗ trợ cả hai dạng. Bật GPU T4 và Internet (chỉ để tải upstream pinned và ninja). Chạy lần lượt; giữ các artifact fail/OOM, không sửa recipe để cứu gate.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
INPUT = Path('/kaggle/input')
archives = list(INPUT.rglob('stage1_source_bundle.zip'))
trees = sorted({path.parent for path in INPUT.rglob('bundle_manifest.json')})
assert len(archives) + len(trees) == 1, (
    f'Expected exactly one source archive or extracted tree; archives={archives}, trees={trees}'
)
ROOT = Path('/kaggle/working/NestedLearningForCAD-stage1')
if ROOT.exists():
    shutil.rmtree(ROOT)
if archives:
    ROOT.mkdir()
    with zipfile.ZipFile(archives[0]) as archive:
        for name in archive.namelist():
            assert (ROOT / name).resolve().is_relative_to(ROOT.resolve())
        archive.extractall(ROOT)
else:
    shutil.copytree(trees[0], ROOT)
manifest = json.loads((ROOT/'bundle_manifest.json').read_text())
for name, expected in manifest['files'].items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest() == expected, name
os.chdir(ROOT)
OUT = Path('/kaggle/working/stage1_evidence')
OUT.mkdir(exist_ok=True)
(OUT/'bundle_manifest.json').write_text(json.dumps(manifest, indent=2))
print(manifest['origin'])
import torch
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
def run(script, *args):
    command = [sys.executable, '-B', f'scripts/hope_cad_v2/{script}', *map(str,args)]
    print(' '.join(command), flush=True)
    result = subprocess.run(command)
    with (OUT/'commands.jsonl').open('a') as log:
        log.write(json.dumps({'command': command, 'returncode': result.returncode})+'\n')
    return result.returncode

assert run('verify_stage1.py', '--output', OUT/'stage1_verification_kaggle.json') == 0
assert run('diagnose_visual.py', '--device', 'cuda', '--output', OUT/'stage1_contraction_kaggle.json') == 0


In [ ]:
# Chính xác 16 support events, canonical dimensions, create_graph=True.
# Script records OOM/nonfinite; do not shrink support or detach history.
resource_status = run('profile_stage1.py', '--device', 'cuda', '--output', OUT/'stage1_canonical_profile_kaggle.json')
print('Resource gate return code:', resource_status)


In [ ]:
# Upstream source verified against pinned per-file hashes by the harness.
import urllib.request
subprocess.run([sys.executable, '-m', 'pip', 'install', 'ninja'], check=True)
PIN = '98cfb5d6004472a0ba3d212d3cf884393e39c134'
archive_path = Path('/kaggle/working/visionhope_pinned.zip')
urllib.request.urlretrieve(f'https://codeload.github.com/PSRben/VisionHOPE/zip/{PIN}', archive_path)
upstream_parent = Path('/kaggle/working/visionhope_pinned')
upstream_parent.mkdir(exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    for name in archive.namelist():
        assert (upstream_parent/name).resolve().is_relative_to(upstream_parent.resolve())
    archive.extractall(upstream_parent)
os.environ['MAX_JOBS'] = '2'
parity_status = run('verify_upstream.py', '--upstream', upstream_parent/f'VisionHOPE-{PIN}', '--output', OUT/'stage1_upstream_parity_kaggle.json')
print('Upstream gate return code:', parity_status)


In [ ]:
# Download this archive even if a gate fails; failures are evidence.
import shutil
evidence_zip = shutil.make_archive('/kaggle/working/stage1_evidence', 'zip', OUT)
print(evidence_zip)
from IPython.display import FileLink, display
display(FileLink(evidence_zip))
